# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI and relabeling sequences. 

In [1]:
# Housekeeping

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [2]:
# Paths and input

# Input
browser = input("Browser (Firefox, Chrome, or Edge): ")
sleep_time = input("Seconds to wait in between clicks (recommended 3): ")
# locations = input("Locations (separate with commas and no spaces in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date: (format: MM-DD-YYYY): ")

# Dates and locations
start_date = "11-01-2021"
end_date = "12-05-2025"
date_range = start_date + "--" + end_date
locations = "Antarctica,North America,South America"

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" # "_Antarctica_North_America_South_America/"
andersen = home + "Andersen/avian-influenza/metadata/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

genotypes_df = pd.read_excel("genotype_key.xlsx")
genotypes = list(genotypes_df["Genotype"])

# serotype = "H5N1"
# genotypes = ["B3.13", "D1.1", "D1.3"]
# genotypes = ["B3.2", "B3.6", "B3.7", "B3.5", "A3", "B3.13", "D1.1", "D1.3"]


## Downloading Data

In [3]:


# browser = "Chrome"
# sleep_time = "3"
# locations = "South America"
# start_date = "07-01-2025"
# end_date = "07-25-2025"



In [4]:
os.chdir(downloads)

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

    # Move downloaded files to saved downloads
    for dirpath, dirs, files in os.walk(downloads):
        if len(files) > 0: # If we have any files that need to be moved
            for file in files:
                file_name = os.path.join(dirpath, file)
                destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                try:
                    shutil.move(file_name, destination_path)
                except:
                    print("Error moving file", file_name)
                    continue 
        else: # If we don't have any downloaded files
            # Get files
            open_ncbi_virus(browser, sleep_time, locations, start_date, end_date)

            # Re-try 
            for dirpath, dirs, files in os.walk(downloads):
                if len(files) > 0: # If we have any files that need to be moved
                    for file in files:
                        file_name = os.path.join(dirpath, file)
                        destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                        try:
                            shutil.move(file_name, destination_path)
                        except:
                            print("Error moving file", file_name)
                            continue 
                break 
        break 

## De-Duplication

In [263]:
os.chdir(downloads_saved)
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.Isolate, as_index=False).size()
# print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="Isolate")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
metadata_segments.head()

118144


,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Host,Tissue_Specimen_Source,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size
0,PV659823.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Cygnus atratus,NaN,"Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,1.0,2025-05-12,2025-12-04,ssRNA(-),8
1,PV659824.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Cygnus atratus,NaN,"Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,1.0,2025-05-12,2025-12-04,ssRNA(-),8
2,PV659825.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Cygnus atratus,NaN,"Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,1.0,2025-05-12,2025-12-04,ssRNA(-),8
3,PV659826.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Cygnus atratus,NaN,"Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,1.0,2025-05-12,2025-12-04,ssRNA(-),8
4,PV659827.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Cygnus atratus,NaN,"Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,1.0,2025-05-12,2025-12-04,ssRNA(-),8


In [264]:
# # De-duplicate from Andersen using SRA Accession

# # If even one SRA Accession in this list exists in the NCBI Virus dataframe, remove it from NCBI Virus dataframe
# andersen_sras = []
# # Grab files
# for dirpath, dirs, files in os.walk(andersen):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         if ".fasta" in file_name:
#             fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
#             sra_accessions = fasta_file["Identifier"]
#             for value in sra_accessions.values:
#                 if "SRR" in value:
#                     andersen_sras.append(value)
#             isolates = fasta_file["Isolate_Id"]
#             for value in isolates.values:
#                 andersen_sras.append(value)
#             partials = fasta_file["Partials"]
#             for value in partials.values:
#                 andersen_sras.append(value)
#     break 

# # Grab more files
# # for dirpath, dirs, files in os.walk(andersen_dedup):
# #     for file in files:
# #         file_name = os.path.join(dirpath, file)
# #         if ".fasta" in file_name:
# #             fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
# #             sra_accessions = fasta_file["Identifier"]
# #             for value in sra_accessions.values:
# #                 if "SRR" in value:
# #                     andersen_sras.append(value)
            
# metadata_segments["Partials"] = metadata_segments["Isolate"].apply(lambda x: partial_isolate(x) if x == x else x)
# # Remove duplicates from Andersen
# for value in andersen_sras: # to remove
#     if "SRR" in value:
#         metadata_segments = metadata_segments[metadata_segments["SRA_Accession"] != value]
    
#     metadata_segments = metadata_segments[metadata_segments["Isolate"] != value]
#     metadata_segments = metadata_segments[metadata_segments["Partials"] != value]
#     # print(value)
    
# print(len(metadata_segments))
# # metadata_segments
# # print(count)

## Add sequences to dataframe

In [265]:
# NCBI Virus Naming Convention Example:
# Influenza A virus |USA: IN|25-006338-001-original|H5N1|2025-02-20|Meleagris gallopavo|GenBank|SRR33124721|SAMN47941411|PRJNA980729|Influenza A virus (A/Turkey/IN/25-006338-001-original/2025(H5N1)) segment 1 polymerase PB2 (PB2) gene, complete cds
# Organism_Name | Geo_Location | Isolate | Genotype | Collection_Date | Host | GenBank_RefSeq | SRA_Accession | BioSample | BioProject | GenBank_Title | Accession

# Get sequences and headers together
# headers = []
# isolates = []
# sras = []
# headers_seqs = {}

os.chdir(downloads_saved)

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

# Filter dataframe to only include filtered SRA accessions
# sequences_fasta = sequences_fasta[sequences_fasta["full_header"].str.contains("|".join(list(metadata_segments["SRA_Accession"].values)))]
# sequences_fasta["SRA_Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-4])
sequences_fasta["Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-1][:-1])

# Extract segment number so that we can add the correct sequences to the correct sample
sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-2]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
print(sequences_fasta.head())
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on=["Accession", "Segment"])

118144
                                         full_header  \
0  >Influenza A virus |Brazil: SapucaiaSulBR|1246...   
1  >Influenza A virus |Brazil: SapucaiaSulBR|1246...   
2  >Influenza A virus |Brazil: SapucaiaSulBR|1246...   
3  >Influenza A virus |Brazil: SapucaiaSulBR|1246...   
4  >Influenza A virus |Brazil: SapucaiaSulBR|1246...   

                                            sequence   Accession  Segment  
0  AGCRAAAGCAGGTCAAATATATTCAATATGGAGAGAATCAAAGAAT...  PV659823.1        1  
1  AGCRAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...  PV659824.1        2  
2  AGCRAAAGCAGGTACTGATCCAAAATGGAAGACTTTGTGCGACAAT...  PV659825.1        3  
3  AGCAAAAGCAGGGGTTCATTCTGTCAAAATGGAGAACATAGTACTA...  PV659826.1        4  
4  AGCAAAAGCAGGGTAGATAATCACTCACTGAGTGACATCCACATCA...  PV659827.1        5  
101856


In [266]:
# # Make sure H5N1 is the only serotype we have

# metadata_segments["serotype"] = metadata_segments["full_header"].apply(lambda x: re.search(r'H.N.', x).group(0))

# metadata_segments = metadata_segments[metadata_segments["serotype"] == serotype]

## Find genotype using old genoflu results or Andersen Lab genoflu output

In [268]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,PV659823.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,1.0,2025-05-12,2025-12-04,ssRNA(-),8,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCRAAAGCAGGTCAAATATATTCAATATGGAGAGAATCAAAGAAT...
1,PV659824.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,1.0,2025-05-12,2025-12-04,ssRNA(-),8,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCRAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...
2,PV659825.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,1.0,2025-05-12,2025-12-04,ssRNA(-),8,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCRAAAGCAGGTACTGATCCAAAATGGAAGACTTTGTGCGACAAT...
3,PV659826.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,1.0,2025-05-12,2025-12-04,ssRNA(-),8,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGGGTTCATTCTGTCAAAATGGAGAACATAGTACTA...
4,PV659827.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Rivetti,A.V. Jr., Reischak,D., Carnegie,L., Ot...","Ministerio da Agricultura e Pecuaria, Laborato...",Brazil,1.0,2025-05-12,2025-12-04,ssRNA(-),8,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGGTAGATAATCACTCACTGAGTGACATCCACATCA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101851,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...
101852,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...
101853,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...
101854,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Youk,S., Leyson,C., Parris,D., Suarez,D., Pant...","USDA ARS SEPRL, Exotic & Emerging Avian Viral ...",USA,NaN,1995,2021-11-01,ssRNA(-),8,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...


In [269]:
genoflu_old = pd.read_csv("output.tsv", delimiter="\t")
genoflu_old = genoflu_old.rename(columns={"Strain":"Partial_Header"})
# print(genoflu_old)
metadata_segments["Partial_Header"] = metadata_segments["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))
metadata_segments["Partial_Header"] = metadata_segments["Partial_Header"].apply(lambda x: x.replace(">", ""))
for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
    metadata_segments["Partial_Header"] = metadata_segments["Partial_Header"].apply(lambda x: x.replace(c, "_"))
# metadata_segments["Accession_Root"] = metadata_segments["Accession"].values[0].split(".")[0]
# print(metadata_segments["Partial_Header"])

metadata_segments_old = metadata_segments.merge(genoflu_old, how="inner", on="Partial_Header") # .dropna(subset="SRA_Accession")
metadata_segments_new = metadata_segments.merge(genoflu_old,indicator = True, how='left', on="Partial_Header").loc[lambda x : x['_merge']!='both'] # .dropna(subset="SRA_Accession")

print(len(metadata_segments_old))
print(len(metadata_segments_new))

101992
168


In [270]:
metadata_segments_new

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,sequence,Partial_Header,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Date run,_merge
8,PV660431.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATCAAAGAAT...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
9,PV660432.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
10,PV660433.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,AGCAAAAGCAGGTACTGATCCAAAATGGAAGACTTTGTGCGACAAT...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
11,PV660434.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,AGCAAAAGCAGGGGTTCATTCTGTCAAAATGGAGAACATAGTACTA...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
12,PV660435.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,AGCAAAAGCAGGGTAGATAATCACTCACTGAGTGACATCCACATCA...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100575,OQ747867.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGAGAGAATAAAAGAACTAAGAGATCTAATGTCACAGTCTCGCA...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
100579,OQ747877.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...,Influenza_A_virus__Peru_A273_H5N1_2022_12__Gen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
100583,OQ747893.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
100595,OQ747888.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGATTCCAACACTGTGTCAAGCTTTCAGGTAGACTGCTTTCTTT...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [281]:
os.chdir(andersen)

genoflu_andersen = pd.read_csv("genoflu_results.tsv", delimiter="\t")
genoflu_andersen = genoflu_andersen.rename(columns={"sample":"SRA_Accession"})

metadata_segments_known_andersen = metadata_segments_new.merge(genoflu_andersen, how="inner", on="SRA_Accession")
print(metadata_segments_known_andersen)
metadata_segments_known = pd.merge(metadata_segments_old, metadata_segments_known_andersen, how="left") # .drop_duplicates(keep="last", inplace=True) # Since we know both of these

print(genoflu_andersen)

    Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0  PQ135274.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
1  PQ135275.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
2  PQ135276.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
3  PQ135277.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
4  PQ135278.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
5  PQ135279.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
6  PQ135280.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
7  PQ135281.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   

     BioProject      Organism_Name                         Species  \
0  PRJNA1102327  Influenza A virus  Alphainfluenzavirus influenzae   
1  PRJNA1102327  Influenza A virus  Alphainfluenzavirus influenzae   
2  PRJNA1102327  Influenza A virus  Alphainfluenzavirus influenzae   
3  PRJNA1102327  Influenza

In [282]:
# metadata_segments_known = metadata_segments_known.dropna(subset="SRA_Accession")

metadata_segments_known["Genotype_y"]

0         Not assigned: Only 4 segments >98.0% match fou...
1         Not assigned: Only 4 segments >98.0% match fou...
2         Not assigned: Only 4 segments >98.0% match fou...
3         Not assigned: Only 4 segments >98.0% match fou...
4         Not assigned: Only 4 segments >98.0% match fou...
                                ...                        
101987    Not assigned: Only 0 segments >98.0% match fou...
101988    Not assigned: Only 0 segments >98.0% match fou...
101989    Not assigned: Only 0 segments >98.0% match fou...
101990    Not assigned: Only 0 segments >98.0% match fou...
101991    Not assigned: Only 0 segments >98.0% match fou...
Name: Genotype_y, Length: 101992, dtype: object

In [ ]:
# metadata_segments_unknown = metadata_segments_new.drop("_merge", axis=1).merge(metadata_segments_known.drop("_merge", axis=1), indicator = True, how='left', on="SRA_Accession").loc[lambda x : x['_merge']!='both']
# metadata_segments_unknown = metadata_segments_new[~metadata_segments_new["SRA_Accession"].apply(tuple,1).isin(metadata_segments_known["SRA_Accession"].apply(tuple,1))]
# metadata_segments_unknown = pd.concat([metadata_segments_new, metadata_segments_known]).drop_duplicates(subset=["Accession", "Segment"], keep="last")


In [ ]:
metadata_segments_unknown = metadata_segments_new


In [293]:

metadata_segments_known["Genotype_y"]

0         Not assigned: Only 4 segments >98.0% match fou...
1         Not assigned: Only 4 segments >98.0% match fou...
2         Not assigned: Only 4 segments >98.0% match fou...
3         Not assigned: Only 4 segments >98.0% match fou...
4         Not assigned: Only 4 segments >98.0% match fou...
                                ...                        
101987    Not assigned: Only 0 segments >98.0% match fou...
101988    Not assigned: Only 0 segments >98.0% match fou...
101989    Not assigned: Only 0 segments >98.0% match fou...
101990    Not assigned: Only 0 segments >98.0% match fou...
101991    Not assigned: Only 0 segments >98.0% match fou...
Name: Genotype_y, Length: 101992, dtype: object

In [287]:
# metadata_segments_unknown = metadata_segments_unknown.dropna(subset="SRA_Accession")

metadata_segments_unknown

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,sequence,Partial_Header,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Date run,_merge
8,PV660431.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATCAAAGAAT...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
9,PV660432.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
10,PV660433.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,AGCAAAAGCAGGTACTGATCCAAAATGGAAGACTTTGTGCGACAAT...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
11,PV660434.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,AGCAAAAGCAGGGGTTCATTCTGTCAAAATGGAGAACATAGTACTA...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
12,PV660435.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,AGCAAAAGCAGGGTAGATAATCACTCACTGAGTGACATCCACATCA...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100575,OQ747867.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGAGAGAATAAAAGAACTAAGAGATCTAATGTCACAGTCTCGCA...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
100579,OQ747877.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGAAGACTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGC...,Influenza_A_virus__Peru_A273_H5N1_2022_12__Gen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
100583,OQ747893.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
100595,OQ747888.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,ATGGATTCCAACACTGTGTCAAGCTTTCAGGTAGACTGCTTTCTTT...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


## Create FASTA files of unknown genotypes using deduplicated sequences

In [217]:
# # Create 1 fasta file per header
# metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))

# # Create list of dataframes
# df_list = []
# for partial_header in list(set(metadata_segments_unknown["Partial_Header"].values)): # Unique partial headers only
#     # Get smaller dataframe
#     df = metadata_segments_unknown[metadata_segments_unknown["Partial_Header"] == partial_header]
#     df = df.sort_values(by="Segment")
#     # Make sure there are 8 segments
#     if len(df) == 8:
#         # Forbidden characters in headers
#         for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
#             df["full_header"] = df["full_header"].apply(lambda x: x.replace(c, "_"))
#             df_list.append(df)

# # Make fasta files
# for df in df_list:
#     # Forbidden characters in file name 
#     title = df["Accession"].values[0].split(".")[0]
#     # for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
#     #     title = title.replace(c, "_")
#     df_to_fasta(df, title + ".fasta", temp_files)

In [288]:
# Create 8 fasta files per segment

# Get all the segments
metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))

# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments_unknown["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments_unknown[metadata_segments_unknown["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # print(partial_header)
    # print(df["Segment"])
    # break 
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            # if c == "-":
        df["full_header"] = df["Partial_Header"]
        df_list.append(df)

print(metadata_segments_unknown["Partial_Header"])

print(df_list[0]["full_header"])

# Make fasta files
for df in df_list:
    # Forbidden characters in file name 
    # title = df["Accession"].values[0].split(".")[0]
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    # print(df)
    # print(segments)
    # break 
    # for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
    #     title = title.replace(c, "_")
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        # print(one_row["full_header"])
        # print(segment)
        # break 
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)
    # break 

8         >Influenza A virus |Brazil: SapucaiaSulBR|1246...
9         >Influenza A virus |Brazil: SapucaiaSulBR|1246...
10        >Influenza A virus |Brazil: SapucaiaSulBR|1246...
11        >Influenza A virus |Brazil: SapucaiaSulBR|1246...
12        >Influenza A virus |Brazil: SapucaiaSulBR|1246...
                                ...                        
100575    >Influenza A virus |Peru|A273|H5N1|2022-12|Fal...
100579    >Influenza A virus |Peru|A273|H5N1|2022-12||Ge...
100583    >Influenza A virus |Peru|A273|H5N1|2022-12|Fal...
100595    >Influenza A virus |Peru|A273|H5N1|2022-12|Fal...
100599    >Influenza A virus |Peru|A273|H5N1|2022-12|Fal...
Name: Partial_Header, Length: 168, dtype: object
24    >Influenza_A_virus__Brazil__MateusLemeBR_1367_...
25    >Influenza_A_virus__Brazil__MateusLemeBR_1367_...
26    >Influenza_A_virus__Brazil__MateusLemeBR_1367_...
27    >Influenza_A_virus__Brazil__MateusLemeBR_1367_...
28    >Influenza_A_virus__Brazil__MateusLemeBR_1367_...
29    >Infl

## Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND NEW GENOTYPES.** Then, make sure "output.tsv" is in the downloads directory.



In [305]:
'''
To run GenoFLU-multi, first change directories (and activate genoflu conda environment):

conda activate genoflu
cd GenoFLU-multi

And then call the python script:

python bin/genoflu-multi.py -f <FASTA_directory>
'''

'\nTo run GenoFLU-multi, first change directories (and activate genoflu conda environment):\n\nconda activate genoflu\ncd GenoFLU-multi\n\nAnd then call the python script:\n\npython bin/genoflu-multi.py -f <FASTA_directory>\n'

In [306]:
metadata_segments_unknown

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Partial_Header,Genotype_y,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List,Date run,_merge,Strain
8,PV660431.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...
9,PV660432.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...
10,PV660433.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...
11,PV660434.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...
12,PV660435.1,GenBank,GCA_053847005.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100575,OQ747867.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...
100579,OQ747877.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12__Gen...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12__Gen...
100583,OQ747893.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...
100595,OQ747888.1,GenBank,NaN,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only,Influenza_A_virus__Peru_A273_H5N1_2022_12_Falc...


In [307]:
# Merging

os.chdir(downloads_saved)

# Read in genoflu results
output_genoflu = pd.read_csv("results.tsv", delimiter="\t")

metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["full_header"].apply(lambda x: "|".join(x.split("|")[:-2]))
for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
    metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["Partial_Header"].apply(lambda x: x.replace(c, "_"))

metadata_segments_unknown["Partial_Header"] = metadata_segments_unknown["Partial_Header"].apply(lambda x: x.replace(">", ""))

metadata_segments_unknown["Strain"] = metadata_segments_unknown["Partial_Header"]

metadata_genoflu = metadata_segments_unknown.merge(output_genoflu, how="left", on="Strain", suffixes=('_left', '_right')) 

# Fill the rest of the 8 segments with the same genotype
metadata_genoflu = metadata_genoflu.ffill()

print(metadata_genoflu)

print(metadata_genoflu["Genotype_x"])

print(metadata_genoflu.columns)


      Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0    PV660431.1        GenBank  GCA_053847005.1           NaN           NaN   
1    PV660432.1        GenBank  GCA_053847005.1           NaN           NaN   
2    PV660433.1        GenBank  GCA_053847005.1           NaN           NaN   
3    PV660434.1        GenBank  GCA_053847005.1           NaN           NaN   
4    PV660435.1        GenBank  GCA_053847005.1           NaN           NaN   
..          ...            ...              ...           ...           ...   
163  OQ747867.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
164  OQ747877.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
165  OQ747893.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
166  OQ747888.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   
167  OQ747899.1        GenBank  GCA_041198955.1   SRR29694483  SAMN42278987   

       BioProject      Organism_Name               

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_9864\3408649063.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  metadata_genoflu = metadata_genoflu.ffill()


# Concatenate with known genotypes

In [308]:
metadata_segments_known["Genotype_official"] = metadata_segments_known["Genotype_y"]
metadata_segments_known["Serotype"] = metadata_segments_known["Genotype_x"]
metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype"]
metadata_genoflu["Serotype"] = metadata_genoflu["Genotype_x"]

metadata_genoflu = pd.concat([metadata_genoflu, metadata_segments_known])

metadata_genoflu.columns

Index(['Accession', 'GenBank_RefSeq', 'Assembly', 'SRA_Accession', 'BioSample',
       'BioProject', 'Organism_Name', 'Species', 'Genus', 'Family',
       'Genotype_x', 'Isolate', 'Segment', 'GenBank_Title', 'Length',
       'Nuc_Completeness', 'Geo_Location', 'Country', 'USA', 'Host',
       'Tissue_Specimen_Source', 'Submitters', 'Organization', 'Org_location',
       'Publications', 'Collection_Date', 'Release_Date', 'Molecule_type',
       'size', 'full_header', 'sequence', 'Partial_Header', 'Genotype_y',
       'Genotype List Used, >=98.0%_left', 'Genotype Sample Title List_left',
       'Genotype Percent Match List_left', 'Genotype Mismatch List_left',
       'Genotype Average Depth of Coverage List_left', 'Date run_left',
       '_merge', 'Strain', 'Genotype', 'Genotype List Used, >=98.0%_right',
       'Genotype Sample Title List_right', 'Genotype Percent Match List_right',
       'Genotype Mismatch List_right',
       'Genotype Average Depth of Coverage List_right', 'Date run_

In [309]:
metadata_segments_known # .dropna(subset="SRA_Accession")

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,date,File Name,Genotype,"Genotype List Used, >=98.0%_y",Genotype Sample Title List_y,Genotype Percent Match List_y,Genotype Mismatch List_y,Genotype Average Depth of Coverage List_y,Genotype_official,Serotype
0,PV659823.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N1
1,PV659824.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N1
2,PV659825.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N1
3,PV659826.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N1
4,PV659827.1,GenBank,GCA_053846995.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 4 segments >98.0% match fou...,H5N1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101987,OK205883.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2
101988,OK205884.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2
101989,OK205885.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2
101990,OK205886.1,GenBank,GCA_038165665.1,NaN,NaN,NaN,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not assigned: Only 0 segments >98.0% match fou...,H5N2


In [310]:
metadata_genoflu = metadata_genoflu[["Accession", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype_official", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "File Name", "Partial_Header", "Strain"]]

metadata_genoflu["genbank_name"] = metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1] )
metadata_genoflu["Host"] = metadata_genoflu["genbank_name"].apply(lambda x: x.split("/")[1])

# metadata_genoflu = metadata_genoflu.dropna(subset="SRA_Accession")
# metadata_genoflu = metadata_genoflu[~metadata_genoflu.duplicated(["Segment", "SRA_Accession"], keep=False).groupby(df["SRA_Accession"]).transform('sum').ge(8)]
# metadata_genoflu = metadata_genoflu.drop_duplicates(subset=["SRA_Accession", "Segment"], keep="first", inplace=True)

metadata_genoflu # [metadata_genoflu["Genotype"]  == "B3.13"]

,Accession,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype_official,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,Strain,genbank_name
0,PV660431.1,Influenza A virus (A/Cygnus melancoryphus/Sapu...,Cygnus melancoryphus,2025-05-12,NaN,1246-N7-2025,Not assigned: Only 4 segments >98.0% match fou...,Brazil: SapucaiaSulBR,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATCAAAGAAT...,H5N1,1,NaN,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,A/Cygnus melancoryphus/SapucaiaSulBR/1246-N7-2...
1,PV660432.1,Influenza A virus (A/Cygnus melancoryphus/Sapu...,Cygnus melancoryphus,2025-05-12,NaN,1246-N7-2025,Not assigned: Only 4 segments >98.0% match fou...,Brazil: SapucaiaSulBR,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...,H5N1,2,NaN,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,A/Cygnus melancoryphus/SapucaiaSulBR/1246-N7-2...
2,PV660433.1,Influenza A virus (A/Cygnus melancoryphus/Sapu...,Cygnus melancoryphus,2025-05-12,NaN,1246-N7-2025,Not assigned: Only 4 segments >98.0% match fou...,Brazil: SapucaiaSulBR,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGTACTGATCCAAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,NaN,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,A/Cygnus melancoryphus/SapucaiaSulBR/1246-N7-2...
3,PV660434.1,Influenza A virus (A/Cygnus melancoryphus/Sapu...,Cygnus melancoryphus,2025-05-12,NaN,1246-N7-2025,Not assigned: Only 4 segments >98.0% match fou...,Brazil: SapucaiaSulBR,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGGGTTCATTCTGTCAAAATGGAGAACATAGTACTA...,H5N1,4,NaN,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,A/Cygnus melancoryphus/SapucaiaSulBR/1246-N7-2...
4,PV660435.1,Influenza A virus (A/Cygnus melancoryphus/Sapu...,Cygnus melancoryphus,2025-05-12,NaN,1246-N7-2025,Not assigned: Only 4 segments >98.0% match fou...,Brazil: SapucaiaSulBR,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGGTAGATAATCACTCACTGAGTGACATCCACATCA...,H5N1,5,NaN,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,A/Cygnus melancoryphus/SapucaiaSulBR/1246-N7-2...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101987,OK205883.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Not assigned: Only 0 segments >98.0% match fou...,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...,H5N2,4,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995
101988,OK205884.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Not assigned: Only 0 segments >98.0% match fou...,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...,H5N2,5,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995
101989,OK205885.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Not assigned: Only 0 segments >98.0% match fou...,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...,H5N2,6,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995
101990,OK205886.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,NaN,28159-398,Not assigned: Only 0 segments >98.0% match fou...,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...,H5N2,7,NaN,Influenza_A_virus__Mexico__Veracruz_28159_398_...,NaN,A/chicken/Veracruz/28159-398/1995


In [311]:
metadata_genoflu["Serotype"]

0         H5N1
1         H5N1
2         H5N1
3         H5N1
4         H5N1
          ... 
101987    H5N2
101988    H5N2
101989    H5N2
101990    H5N2
101991    H5N2
Name: Serotype, Length: 102160, dtype: object

We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [313]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['dunlin', 'snowy egret', 'coopers hawk', 'bald eagle', 'barred owl', 'fisher', 'alpaca', 'sula leucogaster', 'michigan', 'fancy chicken', 'western snowy plover', 'american crow', 'wild bird', 'blue-winged and cinnamon teal', 'hooded merganser', 'glaucous gull', 'south american sea lion', 'backyard chicken', 'nasua nasua', 'lesser white-fronted goose', 'raccoon', 'cormorant', 'otaria flavescens', 'merlin', 'eurasian collared dove', 'franklins gull', 'burrowing owl', 'cinnamon teal', 'cattle milk product', 'crested caracara', 'sanderling', 'great-tailed grackle', 'grackle', 'wild duck', 'eastern screech owl', 'emperor goose', 'vulture', 'american white pelican', 'mountain lion', 'chukar', 'gadwall duck', 'ostrich', 'numida meleagris', 'gallus gallus', 'skunk', 'striped skunk', 'great shearwater', 'common murre', 'teal', 'sparrow', 'iceland gull', 'willet', 'unknown', 'wood duck', 'turkey vulture', 'pelican', 'blue-winged teal', 'anatinae', 'green-winged teal', 'great grey owl', 'dog', '

In [318]:
metadata_genoflu["Genotype_official"] = metadata_genoflu["Genotype_official"].apply(lambda x: "Not assigned" if "Not assigned" in str(x) else x)
metadata_genoflu["Genotype"] = metadata_genoflu["Genotype_official"]

In [319]:
metadata_genoflu

,Accession,GenBank_Title,Host,Collection_Date,SRA_Accession,Isolate,Genotype_official,Geo_Location,full_header,sequence,Serotype,Segment,File Name,Partial_Header,Strain,genbank_name,Host_Type,Years,Geo_Location_Abrv,Genotype
0,PV660431.1,Influenza A virus (A/Cygnus melancoryphus/Sapu...,cygnus melancoryphus,2025-05-12,,1246-N7-2025,Not assigned,Brazil: SapucaiaSulBR,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGTCAAATATATTCAATATGGAGAGAATCAAAGAAT...,H5N1,1,,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,A/Cygnus melancoryphus/SapucaiaSulBR/1246-N7-2...,wild_avian,2025,Brazil-ES,Not assigned
1,PV660432.1,Influenza A virus (A/Cygnus melancoryphus/Sapu...,cygnus melancoryphus,2025-05-12,,1246-N7-2025,Not assigned,Brazil: SapucaiaSulBR,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGCAAACCATTTGAATGGATGTCAATCCGACTTTAC...,H5N1,2,,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,A/Cygnus melancoryphus/SapucaiaSulBR/1246-N7-2...,wild_avian,2025,Brazil-ES,Not assigned
2,PV660433.1,Influenza A virus (A/Cygnus melancoryphus/Sapu...,cygnus melancoryphus,2025-05-12,,1246-N7-2025,Not assigned,Brazil: SapucaiaSulBR,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGTACTGATCCAAAATGGAAGACTTTGTGCGACAAT...,H5N1,3,,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,A/Cygnus melancoryphus/SapucaiaSulBR/1246-N7-2...,wild_avian,2025,Brazil-ES,Not assigned
3,PV660434.1,Influenza A virus (A/Cygnus melancoryphus/Sapu...,cygnus melancoryphus,2025-05-12,,1246-N7-2025,Not assigned,Brazil: SapucaiaSulBR,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGGGTTCATTCTGTCAAAATGGAGAACATAGTACTA...,H5N1,4,,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,A/Cygnus melancoryphus/SapucaiaSulBR/1246-N7-2...,wild_avian,2025,Brazil-ES,Not assigned
4,PV660435.1,Influenza A virus (A/Cygnus melancoryphus/Sapu...,cygnus melancoryphus,2025-05-12,,1246-N7-2025,Not assigned,Brazil: SapucaiaSulBR,>Influenza A virus |Brazil: SapucaiaSulBR|1246...,AGCAAAAGCAGGGTAGATAATCACTCACTGAGTGACATCCACATCA...,H5N1,5,,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,Influenza_A_virus__Brazil__SapucaiaSulBR_1246_...,A/Cygnus melancoryphus/SapucaiaSulBR/1246-N7-2...,wild_avian,2025,Brazil-ES,Not assigned
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101987,OK205883.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,,28159-398,Not assigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGGGTATCAGATATCAAAATGGAAAGAATAGTGATT...,H5N2,4,,Influenza_A_virus__Mexico__Veracruz_28159_398_...,,A/chicken/Veracruz/28159-398/1995,domestic_avian,1995,USA-NM,Not assigned
101988,OK205884.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,,28159-398,Not assigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTTAGATAATCACTCACCGAGTGACATTCACATCA...,H5N2,5,,Influenza_A_virus__Mexico__Veracruz_28159_398_...,,A/chicken/Veracruz/28159-398/1995,domestic_avian,1995,USA-NM,Not assigned
101989,OK205885.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,,28159-398,Not assigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGAGTGAAGATGAATCCAAATCAGAAGATAATAACA...,H5N2,6,,Influenza_A_virus__Mexico__Veracruz_28159_398_...,,A/chicken/Veracruz/28159-398/1995,domestic_avian,1995,USA-NM,Not assigned
101990,OK205886.1,Influenza A virus (A/chicken/Veracruz/28159-39...,chicken,1995,,28159-398,Not assigned,Mexico: Veracruz,>Influenza A virus |Mexico: Veracruz|28159-398...,AGCAAAAGCAGGTAGATATTGAAAGATGAGTCTTCTAACCGAGGTC...,H5N2,7,,Influenza_A_virus__Mexico__Veracruz_28159_398_...,,A/chicken/Veracruz/28159-398/1995,domestic_avian,1995,USA-NM,Not assigned


In [320]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location"].apply(lambda x: 
                                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                                        + "-" + 
                                                                        x.split(" ")[-1]
                                                                        if states_ref["Abbreviation"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has the full state name (e.g. "Maryland")
                                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                                        + "-" + 
                                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                                        if states_ref["State"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                                        else 
                                                                        "USA"
                                                                        )

metadata_genoflu["Geo_Location_Abrv"] = metadata_genoflu["Geo_Location_Abrv"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)


# Make new labels
names = ">" + metadata_genoflu["SRA_Accession"].apply(lambda x: x.split(",")[0]) + "|" + metadata_genoflu["genbank_name"] + "|" + metadata_genoflu["Serotype"] + "|" + metadata_genoflu["Geo_Location_Abrv"] + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype"]

metadata_genoflu["Name"] = names

print(metadata_genoflu["Geo_Location_Abrv"])


0         Brazil-ES
1         Brazil-ES
2         Brazil-ES
3         Brazil-ES
4         Brazil-ES
            ...    
101987       USA-NM
101988       USA-NM
101989       USA-NM
101990       USA-NM
101991       USA-NM
Name: Geo_Location_Abrv, Length: 102160, dtype: object


## Rename segments and make complete FASTA files

In [321]:
# Set up segments

segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)
        elif "Not assigned" in genotype:
            print(genotype)
            df = m_g[(m_g["Genotype"].str.contains("Not assigned"))] 
            df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            segment_genotype_dfs.append(df)
            pair = "Unassigned_" + segment
            print(pair)

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_9864\3807190993.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_9864\3807190993.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)


B5.1_PB2
Minor65_PB2
Minor105_PB2
Minor13_PB2
Not assigned
Unassigned_PB2
Minor81_PB2
Minor50_PB2
B3.13_PB2
D1.3_PB2
Minor19_PB2
Minor45_PB2
Minor12_PB2
B1.3_PB2
B3.7_PB2
B4.1_PB2
Minor34_PB2
Minor11_PB2
B3.3_PB2
A4_PB2
Minor63_PB2
Minor87_PB2
B1.2_PB2
Minor77_PB2
B3.2_PB2
B2.2_PB2
Minor90_PB2
B2.1_PB2
Minor102_PB2
A6_PB2
Minor08_PB2
Minor07_PB2
Minor04_PB2
B3.1_PB2
Minor101_PB2
Minor01_PB2
D1.2_PB2
D1.1_PB2
Minor91_PB2
Minor98_PB2
A5_PB2
A1_PB2
C2.1_PB2
B3.6_PB2
B3.10_PB2
B1.1_PB2
Minor14_PB2
B3.4_PB2
B3.5_PB2
A3_PB2
Minor94_PB2
Minor09_PB2
Minor104_PB2
Minor28_PB2
A2_PB2
B3.12_PB2
B5.1_PB1
Minor65_PB1
Minor105_PB1
Minor13_PB1
Not assigned
Unassigned_PB1
Minor81_PB1
Minor50_PB1
B3.13_PB1
D1.3_PB1
Minor19_PB1
Minor45_PB1
Minor12_PB1
B1.3_PB1
B3.7_PB1
B4.1_PB1
Minor34_PB1
Minor11_PB1
B3.3_PB1
A4_PB1
Minor63_PB1
Minor87_PB1
B1.2_PB1
Minor77_PB1
B3.2_PB1
B2.2_PB1
Minor90_PB1
B2.1_PB1
Minor102_PB1
A6_PB1
Minor08_PB1
Minor07_PB1
Minor04_PB1
B3.1_PB1
Minor101_PB1
Minor01_PB1
D1.2_PB1
D1.1_PB

In [322]:
# print(metadata_genoflu[metadata_genoflu["Genotype"] == "B3.2"])

In [323]:
# Create FASTA files

os.chdir(complete_files)

# names = []

for df in segment_genotype_dfs:
    print(df)
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0:
        # print(df)
        if "Not assigned" not in df["Genotype"].values[0]:
            file_name = df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
            output_file = open(complete_files + file_name, "w")

            for index, row in df.iterrows():
                name = df.loc[index, "Name"]
                name = name.replace(" ", "_")
                # names.append(name)
                sequence = df.loc[index, "sequence"]
                # First is header, second is sequence
                output_file.write(name + "\n")
                output_file.write(sequence + "\n")
        else:
            file_name = "Unassigned_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
            output_file = open(complete_files + file_name, "w")

            for index, row in df.iterrows():
                name = df.loc[index, "Name"]
                name = name.replace(" ", "_")
                # names.append(name)
                sequence = df.loc[index, "sequence"]
                # First is header, second is sequence
                output_file.write(name + "\n")
                output_file.write(sequence + "\n")
    output_file.close()

        Accession                                      GenBank_Title  \
19088  PV312885.1  Influenza A virus (A/Great Horned Owl/AB/FAV-0...   

                   Host Collection_Date SRA_Accession           Isolate  \
19088  great horned owl      2022-05-12                FAV-0833-17-2022   

      Genotype_official Geo_Location  \
19088              B5.1   Canada: AB   

                                             full_header  \
19088  >Influenza A virus |Canada: AB|FAV-0833-17-202...   

                                                sequence  ... File Name  \
19088  ATGGAGAGAATAAAAGAGCTAAGAGATTTGATGTCGCAATCTCGCA...  ...             

                                          Partial_Header Strain  \
19088  Influenza_A_virus__Canada__AB_FAV_0833_17_2022...          

                                      genbank_name   Host_Type Years  \
19088  A/Great Horned Owl/AB/FAV-0833-17-2022/2022  wild_avian  2022   

      Geo_Location_Abrv Genotype  \
19088         Canada-AB     B5.1   

In [324]:
# # Concatenate with new Andersen sequences

# os.chdir(combined_files)

# # andersen = home + "Andersen/complete/" + date_range + "/"

# # Andersen files
# filenames_andersen = []
# for genotype in genotypes:
#     # print(gisaid_andersen + genotype.replace(".", "_") + "/")
#     # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
#     for dirpath, dirs, files in os.walk(andersen): # Find the fasta file
#         for file in files:
#             file_name = os.path.join(dirpath, file) # Get file name
#             filenames_andersen.append(file_name)
#         break 

# # NCBI Virus files
# filenames_ncbi = []
# for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
#     for file in files:
#         file_name = os.path.join(dirpath, file) # Get file name
#         filenames_ncbi.append(file_name)
#     break 

# print(filenames_ncbi)

# common_genotypes = set()
# # Concatenate the two -- should not have any overlap due to dates and deduplication 
# for a_file in filenames_andersen:
#     partial_filename_a = a_file.split("_")[-4].split("/")[-1] + "_" + a_file.split("_")[-3]
#     for nv_file in filenames_ncbi:
#         partial_filename_nv = nv_file.split("_")[-3].split("/")[-1] + "_" + nv_file.split("_")[-2]
#         if partial_filename_a == partial_filename_nv:
#             common_genotypes.add(partial_filename_a)
#             filenames = [a_file, nv_file]
#             with open(combined_files + partial_filename_a + "_combined_" + date_range + ".fasta", 'w') as outfile:
#                 for fname in filenames:
#                     with open(fname) as infile:
#                         for line in infile:
#                             outfile.write(line)
#                         infile.close()
#                 outfile.close()

# print(common_genotypes)

# for a_file in filenames_andersen:
#     partial_filename_a = a_file.split("_")[-4].split("/")[-1] + "_" + a_file.split("_")[-3]
#     # If genotype not found in NCBI Virus, include it as well
#     if partial_filename_a not in common_genotypes:
#         with open(combined_files + partial_filename_a + "_combined_" + date_range + ".fasta", 'w') as outfile2:
#             with open(a_file) as infile2:
#                 for line in infile2:
#                     outfile2.write(line)
#                 infile2.close()
#             outfile2.close()